# Análisis de Conversión — The Box (OLA y Montaña S.A.C.)
## Trabajo Final de Ingeniería Empresarial — TFIE 26-2

**Objetivo:** Diagnosticar la baja conversión de visitantes en compradores en las tiendas físicas de The Box, cuantificar la brecha respecto a un referente interno de desempeño y encontrar evidencia que oriente mejoras.

**Fuente única:** `BD The Box Presentacion 30.08.xlsx`  
**Período:** Semanas 1–52 de 2025 y 1–34 de 2026  
**Tiendas:** 16 locales THE BOX en Perú  
**Script de referencia:** `the_box_analisis.R` (valores comentados en R son referencias para contrastar, no resultados verificados)


## Bloque 0 · Setup y configuración

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats
import statsmodels.formula.api as smf

REPO  = Path(r'C:\Users\ADMIN\Desktop\UP\TRABAJO FINAL INGENIERÍA EMPRESARIAL\TFIE_26_2')
DATOS = REPO / 'BD The Box Presentacion 30.08.xlsx'
FIG   = REPO / 'outputs' / 'figuras'
TAB   = REPO / 'outputs' / 'tablas'
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 11, 'axes.labelsize': 10})
sns.set_palette('tab10')

COLS_RAW = ['anio','semana','fecha','inicio_semana','hora','categoria',
            'nro_tienda','tienda','nombre_tienda','flujo','transacciones',
            'venta','unidades']

def savefig(name):
    plt.savefig(FIG / name, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close()

print("Setup OK")
print(f"  pandas {pd.__version__}, numpy {np.__version__}, seaborn {sns.__version__}")
print(f"  FIG -> {FIG}")
print(f"  TAB -> {TAB}")


## Bloque 1 · Objetivo, alcance y diccionario de datos

**Conversión** = Transacciones / Flujo de visitantes (porcentaje de visitantes que realizan una compra)

### Diccionario — Hoja `BD Venta por hora`

| Variable | Tipo | Descripción |
|---|---|---|
| `anio` | int | Año fiscal (2025 o 2026) |
| `semana` | int | Semana del año (1–52) |
| `fecha` | date | Fecha del registro |
| `hora` | int | Hora del día (8–22) |
| `categoria` | str | THE BOX o COOL BRANDS |
| `nombre_tienda` | str | Nombre completo del local |
| `flujo` | int | Visitantes contados por sensor |
| `transacciones` | int | Boletas/tickets emitidos |
| `venta` | float | Venta en soles (S/) |
| `unidades` | int | Unidades vendidas |

**Indicadores derivados:** `conv = trx/flujo`, `ticket = venta/trx`, `upt = unidades/trx`, `ppu = venta/unidades`

### Muestras del análisis

| Etiqueta | Definición |
|---|---|
| `raw` | Todos los registros THE BOX (sin filtro) |
| `dep` | Horario 10–21h, flujo > 0, flujo ≥ transacciones |
| `sw26` | Depurada + 2026 + sem 2–34, f≥100, t≥10 |
| `comp` | Depurada + sem 2–34 (comparación interanual) |
| Comunes | Tiendas presentes en dep 2025 Y 2026, sem 2–34 |


In [ ]:
raw = pd.read_excel(DATOS, sheet_name='BD Venta por hora', usecols=list(range(13)))
raw.columns = COLS_RAW
raw['fecha'] = pd.to_datetime(raw['fecha'], errors='coerce')
for c in ['anio','semana','hora','flujo','transacciones','venta','unidades']:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')

print(f"Shape raw: {raw.shape}   esperado: (122329, 13)")
print(f"Categorías: {raw['categoria'].unique()}")
print(f"Años: {sorted(raw['anio'].dropna().unique().tolist())}")
print(f"Tiendas THE BOX: {raw[raw['categoria']=='THE BOX']['nombre_tienda'].nunique()}")
print()
print(raw.dtypes)


## Bloque 2 · Calidad de datos y depuración

**Pregunta:** ¿Qué proporción de registros presenta inconsistencias y cuánto afectan a la conversión reportada?

**Método:** Estadísticos de calidad por año sobre THE BOX. Cuantificación de exclusiones por cada regla.

**Variables:** `flujo`, `transacciones`, `hora`  |  **Muestra:** `raw` filtrado a THE BOX


In [ ]:
tb = raw[raw['categoria'] == 'THE BOX'].copy()
print(f"Filas THE BOX: {len(tb):,}   esperado: 103 790")

calidad = tb.groupby('anio').apply(lambda g: pd.Series({
    'filas':             len(g),
    'flujo_total':       g['flujo'].sum(),
    'trx_total':         g['transacciones'].sum(),
    'conv_reportada':    g['transacciones'].sum() / g['flujo'].sum(),
    'filas_flujo_cero':  (g['flujo']==0).sum(),
    'trx_en_flujo_cero': g.loc[g['flujo']==0, 'transacciones'].sum(),
    'pct_trx_flujo_cero':g.loc[g['flujo']==0,'transacciones'].sum() / g['transacciones'].sum(),
    'flujo_hora22':      g.loc[g['hora']==22, 'flujo'].sum(),
    'trx_hora22':        g.loc[g['hora']==22, 'transacciones'].sum(),
})).reset_index()

print("\n--- Tabla de calidad por año ---")
print(calidad.to_string(index=False))
print("\nEsperado 2026: conv_reportada≈0.06815, pct_trx_flujo_cero≈0.0484")
print("Esperado 2025: conv_reportada≈0.06468, pct_trx_flujo_cero≈0.0711")
calidad.to_csv(TAB / 'calidad_datos.csv', index=False)


In [ ]:
# Reglas de depuración secuenciales
excl_horario = tb[(tb['hora'] < 10) | (tb['hora'] > 21)]
tb_hor       = tb[(tb['hora'] >= 10) & (tb['hora'] <= 21)]
excl_flujo0  = tb_hor[tb_hor['flujo'] <= 0]
tb_pos       = tb_hor[tb_hor['flujo'] > 0]
excl_incons  = tb_pos[tb_pos['flujo'] < tb_pos['transacciones']]
dep          = tb_pos[tb_pos['flujo'] >= tb_pos['transacciones']].copy()

def rsm(df, label):
    return {'Regla': label, 'Filas': len(df),
            'Flujo': df['flujo'].sum(), 'Transacciones': df['transacciones'].sum(),
            'Venta S/': df['venta'].sum()}

excl_df = pd.DataFrame([
    rsm(excl_horario, 'Fuera de horario 10-21h'),
    rsm(excl_flujo0,  'Flujo = 0'),
    rsm(excl_incons,  'Flujo < Transacciones'),
    rsm(dep,          'MUESTRA DEPURADA (dep)'),
])
print("--- Exclusiones por regla ---")
print(excl_df.to_string(index=False))
excl_df.to_csv(TAB / 'exclusiones_depuracion.csv', index=False)

print(f"\nFilas muestra depurada: {len(dep):,}")
conv_dep = dep.groupby('anio').apply(
    lambda g: g['transacciones'].sum()/g['flujo'].sum()).reset_index(name='conv_depurada')
print("Conversión depurada por año:")
print(conv_dep.to_string(index=False))
print("Esperado 2026: 0.06467  |  2025: 0.05973")


## Bloque 3 · Evolución 2025–2026 (semanas 2–34)

**Pregunta:** ¿Cuánto creció la venta entre años y qué palancas lo explican?

**Método:** Descomposición logarítmica Venta = Flujo × Conversión × UPT × Precio unitario, semanas 2–34.

> La semana 1 se excluye (año nuevo/apertura). La comparación exige que ambos años cubran el mismo rango de semanas.


In [ ]:
comp_all = dep[dep['semana'].between(2,34)].groupby('anio').agg(
    flujo=('flujo','sum'), trx=('transacciones','sum'),
    venta=('venta','sum'), unid=('unidades','sum'),
    tiendas=('nombre_tienda','nunique')).reset_index()
comp_all['conv']   = comp_all['trx']   / comp_all['flujo']
comp_all['ticket'] = comp_all['venta'] / comp_all['trx']
comp_all['upt']    = comp_all['unid']  / comp_all['trx']
comp_all['ppu']    = comp_all['venta'] / comp_all['unid']

print("--- Comparación interanual semanas 2-34 ---")
print(comp_all[['anio','tiendas','flujo','trx','venta','conv','ticket','upt']].to_string(index=False))

a25 = comp_all[comp_all['anio']==2025].iloc[0]
a26 = comp_all[comp_all['anio']==2026].iloc[0]
dlv = np.log(a26['venta'] / a25['venta'])

crec_venta = a26['venta']/a25['venta'] - 1
ap_flujo   = np.log(a26['flujo']  / a25['flujo'])  / dlv
ap_conv    = np.log(a26['conv']   / a25['conv'])   / dlv
ap_ticket  = np.log(a26['ticket'] / a25['ticket']) / dlv

print(f"\nCrecimiento de venta: {crec_venta:+.1%}   esperado: +94.1%")
print(f"  Aporte flujo:       {ap_flujo:.1%}   esperado: 35.9%")
print(f"  Aporte conversión:  {ap_conv:.1%}   esperado: 16.5%")
print(f"  Aporte ticket:      {ap_ticket:.1%}   esperado: 47.5%")
comp_all.to_csv(TAB / 'comparacion_interanual.csv', index=False)


In [ ]:
# Tiendas comunes
t25 = set(dep[(dep['anio']==2025) & dep['semana'].between(2,34)]['nombre_tienda'].unique())
t26 = set(dep[(dep['anio']==2026) & dep['semana'].between(2,34)]['nombre_tienda'].unique())
comunes = sorted(t25 & t26)
print(f"Tiendas comunes: {len(comunes)}")

comp_com = dep[dep['semana'].between(2,34) & dep['nombre_tienda'].isin(comunes)]\
    .groupby('anio').apply(lambda g: pd.Series({
        'venta': g['venta'].sum(),
        'conv':  g['transacciones'].sum()/g['flujo'].sum()})).reset_index()
crec_com = comp_com[comp_com['anio']==2026]['venta'].values[0] / \
           comp_com[comp_com['anio']==2025]['venta'].values[0] - 1
print(f"Crecimiento venta tiendas comunes: {crec_com:+.1%}   esperado: +75.9%")
print(comp_com.to_string(index=False))


In [ ]:
# Gráfico descomposición
aportes = {'Flujo': ap_flujo, 'Conversión': ap_conv, 'Ticket': ap_ticket}
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(aportes.keys(), [v*100 for v in aportes.values()],
              color=['#4C72B0','#DD8452','#55A868'], width=0.5)
for bar, val in zip(bars, aportes.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1%}', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Aporte al crecimiento de venta (%)')
ax.set_title(f'Descomposición del crecimiento de venta 2025→2026\n({crec_venta:+.1%} total, semanas 2-34)')
ax.axhline(0, color='black', lw=0.8)
plt.tight_layout()
savefig('evolucion_descomposicion.png')

# Gráfico conversión semanal
conv_sem = dep[dep['semana'].between(2,34)].groupby(['anio','semana']).apply(
    lambda g: g['transacciones'].sum()/g['flujo'].sum()).reset_index(name='conv')
fig, ax = plt.subplots(figsize=(10, 4))
for anio, grp in conv_sem.groupby('anio'):
    ax.plot(grp['semana'], grp['conv']*100, marker='o', ms=3, label=str(int(anio)))
ax.set_xlabel('Semana del año'); ax.set_ylabel('Conversión (%)')
ax.set_title('Conversión semanal por año (semanas 2-34, muestra depurada)')
ax.legend(title='Año')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
savefig('evolucion_conversion_semanal.png')


## Bloque 4 · Conversión real y referencia interna (frontera)

**Pregunta:** ¿Cuál es la conversión real de la cadena y qué brecha existe respecto al mejor desempeño propio de cada tienda?

**Método:** Percentiles 75, 90 y 100 de conversión semanal propia de cada tienda, ponderados por flujo. El p90 ponderado es la "referencia interna" — **no es una meta sostenible garantizada** (es desempeño observado que puede reflejar condiciones excepcionales).

**Muestra:** `sw26` — tienda×semana 2026, sem 2–34, f≥100 y t≥10.


In [ ]:
sw26 = dep[(dep['anio']==2026) & dep['semana'].between(2,34)]\
    .groupby(['nombre_tienda','semana']).agg(
        f=('flujo','sum'), t=('transacciones','sum'),
        v=('venta','sum'), u=('unidades','sum')).reset_index()
sw26 = sw26[(sw26['f']>=100) & (sw26['t']>=10)].copy()
sw26['conv'] = sw26['t'] / sw26['f']

conv_real = sw26['t'].sum() / sw26['f'].sum()
print(f"Conversión real 2026 (sem 2-34, ponderada por flujo): {conv_real*100:.4f}%")

def frontera(p):
    q = sw26.groupby('nombre_tienda').apply(
        lambda g: pd.Series({'q': g['conv'].quantile(p), 'w': g['f'].sum()})).reset_index()
    return (q['q'] * q['w']).sum() / q['w'].sum()

res_f = []
for p in [0.75, 0.90, 1.00]:
    v = frontera(p)
    res_f.append({'Percentil': f'p{int(p*100)}', 'Conv_%': v*100, 'Brecha_pp': (v-conv_real)*100})
    print(f"  p{int(p*100):3d} = {v*100:.4f}%   brecha = {(v-conv_real)*100:.4f} pp")
print("Esperado: p75=7.1980% (0.7157pp) | p90=7.7848% (1.3025pp) | max=8.5843% (2.1020pp)")
pd.DataFrame(res_f).to_csv(TAB / 'frontera_agregada.csv', index=False)


In [ ]:
ft = sw26.groupby('nombre_tienda').apply(lambda g: pd.Series({
    'flujo_total': g['f'].sum(), 'semanas': len(g),
    'conv_real_%': g['t'].sum()/g['f'].sum()*100,
    'p75_%': g['conv'].quantile(0.75)*100,
    'p90_%': g['conv'].quantile(0.90)*100,
    'max_%': g['conv'].max()*100,
})).reset_index()
ft['brecha_p90_pp'] = ft['p90_%'] - ft['conv_real_%']
ft = ft.sort_values('conv_real_%', ascending=False).reset_index(drop=True)
print(ft[['nombre_tienda','semanas','conv_real_%','p90_%','brecha_p90_pp']].to_string(index=False))
ft.to_csv(TAB / 'frontera_por_tienda.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(ft))
ax.barh(y, ft['p90_%'], color='#4C72B0', alpha=0.35, label='Frontera p90')
ax.barh(y, ft['conv_real_%'], color='#DD8452', label='Conversión real')
ax.set_yticks(y)
ax.set_yticklabels(ft['nombre_tienda'].str.replace('THE BOX ',''), fontsize=8)
ax.set_xlabel('Conversión (%)')
ax.set_title('Conversión real vs Frontera p90 por tienda (2026, sem 2-34)')
ax.axvline(conv_real*100, color='red', ls='--', lw=1, label=f'Promedio cadena {conv_real*100:.2f}%')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
savefig('frontera_conversion.png')


## Bloque 5 · Relación tráfico–conversión

**Pregunta:** ¿Existe una relación sistemática entre el volumen de tráfico y la tasa de conversión?

**Método:** (1) Panel hora con efectos fijos tienda×hora + fecha, errores agrupados por tienda. (2) Panel semana con efectos fijos tienda + periodo. (3) Correlaciones de corte transversal entre tiendas.

**Diferencias Python vs R/fixest:** `linearmodels.PanelOLS` aplica within-transformation para absorber efectos fijos. Para dos efectos (tienda_hora + fecha), se crea el identificador compuesto `tienda_hora` como entidad y `fecha` como tiempo. La convergencia puede diferir levemente del HDFE iterativo de fixest.

> ⚠️ La relación negativa **NO demuestra** falta de personal ni ningún problema operativo específico. Puede reflejar composición distinta de visitantes, errores de medición o congestión real. Los efectos fijos controlan medias fijas pero no eliminan todos los posibles confusores.


In [ ]:
from linearmodels.panel import PanelOLS

h = dep[dep['flujo'] >= 20].copy()
h['conv']       = h['transacciones'] / h['flujo']
h['lf']         = np.log(h['flujo'])
h['tienda_hora'] = h['nombre_tienda'] + '_' + h['hora'].astype(str)
# PanelOLS requiere que el índice de tiempo sea numérico o datetime
h['fecha_ord'] = h['fecha'].map(lambda d: d.toordinal() if pd.notna(d) else np.nan)
h = h.dropna(subset=['fecha_ord','conv','lf'])
h['fecha_ord'] = h['fecha_ord'].astype(int)

hpanel = h.set_index(['tienda_hora', 'fecha_ord'])
hpanel = hpanel[~hpanel.index.duplicated(keep='first')]

m1 = PanelOLS(hpanel['conv'], hpanel[['lf']],
              entity_effects=True, time_effects=True,
              drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)

beta1 = m1.params['lf']
ee1   = m1.std_errors['lf']
p1    = m1.pvalues['lf']
n1    = int(m1.nobs)
ef10  = beta1 * np.log(1.1) * 100

print("=== M1: Panel hora (tienda_hora + fecha) ===")
print(f"  n={n1:,}   beta={beta1:.5f}   ee={ee1:.5f}   p={p1:.4f}")
print(f"  +10% flujo => {ef10:.4f} pp conversión")
print(f"  Esperado: n=38866, beta=-0.01583, ee=0.00271, p<0.001, efecto=-0.1509pp")


In [ ]:
swp = dep.groupby(['nombre_tienda','anio','semana']).agg(
    f=('flujo','sum'), t=('transacciones','sum')).reset_index()
swp = swp[swp['f'] >= 200].copy()
swp['conv']    = swp['t'] / swp['f']
swp['lf']      = np.log(swp['f'])
swp['periodo'] = swp['anio'].astype(str) + '_' + swp['semana'].astype(str)

swpanel = swp.set_index(['nombre_tienda', 'periodo'])
m2 = PanelOLS(swpanel['conv'], swpanel[['lf']],
              entity_effects=True, time_effects=True,
              drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)

beta2 = m2.params['lf']
ee2   = m2.std_errors['lf']
p2    = m2.pvalues['lf']
n2    = int(m2.nobs)

print("=== M2: Panel semana (tienda + periodo) ===")
print(f"  n={n2:,}   beta={beta2:.5f}   ee={ee2:.5f}   p={p2:.4f}")
print(f"  Esperado: n=1228, beta=-0.04897, ee=0.02559, p=0.056")

ct = dep[(dep['anio']==2026) & dep['semana'].between(2,34)]\
    .groupby('nombre_tienda').agg(
        f=('flujo','sum'), t=('transacciones','sum'),
        sem_activas=('semana','nunique')).reset_index()
ct['conv']      = ct['t'] / ct['f']
ct['flujo_sem'] = ct['f'] / ct['sem_activas']

r_p, p_p = stats.pearsonr(ct['flujo_sem'], ct['conv'])
r_s, p_s = stats.spearmanr(ct['flujo_sem'], ct['conv'])
print(f"\nPearson  r={r_p:.3f} p={p_p:.3f}   esperado: r=-0.378, p=0.149")
print(f"Spearman r={r_s:.3f} p={p_s:.3f}   esperado: r=-0.615, p=0.011")

ct2 = ct[~ct['nombre_tienda'].str.contains('SAN ISIDRO|CHIMBOTE')]
r_s2, p_s2 = stats.spearmanr(ct2['flujo_sem'], ct2['conv'])
print(f"Spearman sin rupturas: r={r_s2:.3f} p={p_s2:.3f}   esperado: r=-0.758, p=0.002")

modelos_df = pd.DataFrame([
    {'Modelo':'M1 Panel hora', 'n':n1,'beta':round(beta1,5),'ee':round(ee1,5),'p_valor':round(p1,4),'FE':'tienda_hora+fecha'},
    {'Modelo':'M2 Panel semana','n':n2,'beta':round(beta2,5),'ee':round(ee2,5),'p_valor':round(p2,4),'FE':'tienda+periodo'},
    {'Modelo':'M3 Pearson CT','n':len(ct),'beta':round(r_p,3),'ee':None,'p_valor':round(p_p,3),'FE':'Ninguno'},
    {'Modelo':'M3 Spearman CT','n':len(ct),'beta':round(r_s,3),'ee':None,'p_valor':round(p_s,3),'FE':'Ninguno'},
])
modelos_df.to_csv(TAB / 'modelos_trafico_conversion.csv', index=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(ct['flujo_sem'], ct['conv']*100, s=60, alpha=0.8, zorder=3)
for _, row in ct.iterrows():
    ax.annotate(row['nombre_tienda'].replace('THE BOX ',''),
                (row['flujo_sem'], row['conv']*100), fontsize=6.5, ha='left', va='bottom')
m_lin, b_lin = np.polyfit(ct['flujo_sem'], ct['conv']*100, 1)
xr = np.linspace(ct['flujo_sem'].min(), ct['flujo_sem'].max(), 100)
ax.plot(xr, m_lin*xr+b_lin, 'r--', lw=1.2, label=f'Tendencia (r={r_p:.2f})')
ax.set_xlabel('Flujo semanal promedio (visitantes/semana)')
ax.set_ylabel('Conversión (%)')
ax.set_title('Relación flujo semanal vs conversión por tienda (2026, sem 2-34)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.legend(fontsize=8)
plt.tight_layout()
savefig('scatter_flujo_conversion.png')


## Bloque 6 · Variación dentro y entre tiendas

**Pregunta:** ¿La heterogeneidad de conversión está más entre tiendas o dentro de cada tienda a lo largo del tiempo?

**Método:** R² de OLS con efectos fijos de tienda, semana o ambos. Desviación estándar entre y dentro de tiendas.

**Implicación:** Si la variabilidad *dentro* de tienda > *entre* tiendas, la oportunidad de mejora está en la consistencia operativa, no solo en diferencias estructurales entre locales.


In [ ]:
sw_todas = dep.groupby(['nombre_tienda','anio','semana']).agg(
    f=('flujo','sum'), t=('transacciones','sum')).reset_index()
sw_todas = sw_todas[sw_todas['f'] >= 300].copy()
sw_todas['conv'] = sw_todas['t'] / sw_todas['f']

vr = sw_todas[(sw_todas['anio']==2026) & sw_todas['semana'].between(2,34)].copy()

r2_t = smf.ols('conv ~ C(nombre_tienda)', data=vr).fit().rsquared
r2_s = smf.ols('conv ~ C(semana)',        data=vr).fit().rsquared
r2_a = smf.ols('conv ~ C(nombre_tienda) + C(semana)', data=vr).fit().rsquared

sd_entre  = vr.groupby('nombre_tienda').apply(lambda g: g['t'].sum()/g['f'].sum()).std()
sd_dentro = vr.groupby('nombre_tienda')['conv'].std().mean()

print(f"R² tienda  = {r2_t:.4f}   esperado: 0.332")
print(f"R² semana  = {r2_s:.4f}   esperado: 0.080")
print(f"R² ambos   = {r2_a:.4f}   esperado: 0.420")
print(f"Residual   = {1-r2_a:.4f}   esperado: 0.580")
print(f"\nSD entre tiendas  = {sd_entre:.5f}   esperado: 0.00999")
print(f"SD dentro tienda  = {sd_dentro:.5f}   esperado: 0.01200")
print(f"SD dentro > SD entre: {sd_dentro > sd_entre}")

pd.DataFrame({'Componente':['Tienda','Semana','Ambos','Residual'],
              'R2':[r2_t, r2_s, r2_a, 1-r2_a]})\
  .to_csv(TAB / 'descomposicion_varianza.csv', index=False)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
order = vr.groupby('nombre_tienda')['conv'].median().sort_values(ascending=False).index
sns.stripplot(data=vr, x='nombre_tienda', y='conv', order=order,
              jitter=True, alpha=0.5, size=4, ax=ax)
medias = vr.groupby('nombre_tienda')['conv'].mean()
for i, t in enumerate(order):
    ax.plot(i, medias[t], 'r_', ms=18, mew=2.5)
ax.set_xticklabels([t.replace('THE BOX ','') for t in order],
                   rotation=45, ha='right', fontsize=7)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y,_: f'{y*100:.1f}%'))
ax.set_xlabel('')
ax.set_ylabel('Conversión semanal')
ax.set_title('Distribución de conversión semanal por tienda — 2026\n(rojo: media, puntos: semanas individuales)')
plt.tight_layout()
savefig('varianza_conversion_tienda.png')


## Bloque 7 · Patrones por hora y día de la semana

**Pregunta:** ¿Existen patrones sistemáticos de flujo y conversión según la hora del día y el día de la semana?

**Muestra:** `dep`, año 2026, semanas 2–34.

> ⚠️ Los patrones son descriptivos. La composición de visitantes (propósito de visita) puede explicar parte de los efectos. No permiten concluir causalidad sin datos adicionales.


In [ ]:
d26 = dep[(dep['anio']==2026) & dep['semana'].between(2,34)].copy()
d26['dow'] = pd.to_datetime(d26['fecha']).dt.dayofweek + 1  # 1=lun, 7=dom
dias_label = {1:'Lun',2:'Mar',3:'Mié',4:'Jue',5:'Vie',6:'Sáb',7:'Dom'}

por_hora = d26.groupby('hora').agg(
    flujo=('flujo','sum'), trx=('transacciones','sum'), venta=('venta','sum')).reset_index()
por_hora['share']  = por_hora['flujo'] / por_hora['flujo'].sum()
por_hora['conv']   = por_hora['trx']  / por_hora['flujo']
por_hora['ticket'] = por_hora['venta']/ por_hora['trx']

conv_h10   = por_hora.loc[por_hora['hora']==10,'conv'].values[0]
conv_resto = (por_hora.loc[por_hora['hora']>10,'trx'].sum() /
              por_hora.loc[por_hora['hora']>10,'flujo'].sum())
print(f"Conversión hora 10: {conv_h10*100:.2f}%   esperado: 4.21%")
print(f"Conversión resto:   {conv_resto*100:.2f}%   esperado: 6.54%")
por_hora.to_csv(TAB / 'patron_por_hora.csv', index=False)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.bar(por_hora['hora'], por_hora['flujo'], color='#4C72B0', alpha=0.55, label='Flujo')
ax2.plot(por_hora['hora'], por_hora['conv']*100, color='#DD8452', marker='o', lw=2, label='Conversión')
ax1.set_xlabel('Hora del día')
ax1.set_ylabel('Flujo (visitantes)', color='#4C72B0')
ax2.set_ylabel('Conversión (%)', color='#DD8452')
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax1.set_title('Flujo y conversión por hora del día (2026, sem 2-34)')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper left', fontsize=8)
plt.tight_layout()
savefig('patron_hora.png')


In [ ]:
por_dia = d26.groupby('dow').agg(
    flujo=('flujo','sum'), trx=('transacciones','sum'), venta=('venta','sum')).reset_index()
por_dia['share']  = por_dia['flujo'] / por_dia['flujo'].sum()
por_dia['conv']   = por_dia['trx']  / por_dia['flujo']
por_dia['ticket'] = por_dia['venta']/ por_dia['trx']
por_dia['dia']    = por_dia['dow'].map(dias_label)
dom = por_dia[por_dia['dow']==7]
print(f"Domingo: share={dom['share'].values[0]:.1%}  conv={dom['conv'].values[0]*100:.2f}%")
print("Esperado: share=22.4%, conv=5.86%")
por_dia.to_csv(TAB / 'patron_por_dia.csv', index=False)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.bar(por_dia['dia'], por_dia['flujo'], color='#4C72B0', alpha=0.55, label='Flujo')
ax2.plot(por_dia['dia'], por_dia['conv']*100, color='#DD8452', marker='o', lw=2, label='Conversión')
ax1.set_xlabel('Día de la semana')
ax1.set_ylabel('Flujo', color='#4C72B0')
ax2.set_ylabel('Conversión (%)', color='#DD8452')
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax1.set_title('Flujo y conversión por día de la semana (2026, sem 2-34)')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper left', fontsize=8)
plt.tight_layout()
savefig('patron_dia.png')


In [ ]:
pivot_hd = d26.groupby(['hora','dow']).apply(
    lambda g: g['transacciones'].sum()/g['flujo'].sum()).unstack()
pivot_hd.columns = [dias_label.get(c, str(c)) for c in pivot_hd.columns]
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(pivot_hd*100, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label':'Conversión (%)'})
ax.set_xlabel('Día'); ax.set_ylabel('Hora')
ax.set_title('Mapa de calor: conversión por hora y día (2026, sem 2-34)')
plt.tight_layout()
savefig('heatmap_hora_dia.png')

pivot_th = d26.groupby(['nombre_tienda','hora']).apply(
    lambda g: g['transacciones'].sum()/g['flujo'].sum()).unstack()
pivot_th.index = pivot_th.index.str.replace('THE BOX ','')
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_th*100, annot=True, fmt='.1f', cmap='YlGn', ax=ax,
            cbar_kws={'label':'Conversión (%)'})
ax.set_xlabel('Hora'); ax.set_ylabel('Tienda')
ax.set_title('Mapa de calor: conversión por tienda y hora (2026, sem 2-34)')
plt.tight_layout()
savefig('heatmap_tienda_hora.png')


## Bloque 8 · Posibles cambios de medición en contadores

**Pregunta:** ¿Hay evidencia de cambios bruscos en el flujo de San Isidro y Chimbote que indiquen problemas de medición?

> ⚠️ Un cambio brusco es un *indicio* de problema de medición, **no una confirmación**. No puede descartarse que el tráfico real haya cambiado por factores externos (obras, remodelación, campaña). Se requiere auditoría presencial o registros del proveedor del sensor para confirmar.


In [ ]:
for tienda in ['THE BOX SAN ISIDRO', 'THE BOX MEGA PLAZA CHIMBOTE']:
    si = dep[(dep['nombre_tienda']==tienda) & (dep['anio']==2026)]\
        .groupby('semana').agg(f=('flujo','sum'), t=('transacciones','sum'),
                               v=('venta','sum')).reset_index()
    si['conv'] = si['t'] / si['f']
    short = tienda.replace('THE BOX ','')

    fig, axes = plt.subplots(3,1, figsize=(10,7), sharex=True)
    axes[0].plot(si['semana'], si['f'],  marker='o', ms=4, color='#4C72B0')
    axes[0].set_ylabel('Flujo'); axes[0].set_title(f'{short} — Series semanales 2026')
    axes[1].plot(si['semana'], si['t'],  marker='o', ms=4, color='#DD8452')
    axes[1].set_ylabel('Transacciones')
    axes[2].plot(si['semana'], si['conv']*100, marker='o', ms=4, color='#55A868')
    axes[2].set_ylabel('Conversión (%)'); axes[2].set_xlabel('Semana')
    axes[2].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
    plt.tight_layout()
    fname = 'serie_' + tienda.lower().replace('the box ','').replace(' ','_') + '.png'
    savefig(fname)
    print(f"\n--- {tienda} ---")
    print(si[['semana','f','t','conv']].to_string(index=False))


In [ ]:
# Comparación antes / después para San Isidro
si_full = dep[(dep['nombre_tienda']=='THE BOX SAN ISIDRO') & (dep['anio']==2026)]\
    .groupby('semana').agg(f=('flujo','sum'), t=('transacciones','sum'),
                           v=('venta','sum')).reset_index()
si_full['conv'] = si_full['t'] / si_full['f']

for tramo, mask in [('sem 2-19',  si_full['semana'].between(2,19)),
                    ('sem 22-34', si_full['semana'].between(22,34))]:
    g = si_full[mask]
    if len(g) == 0: continue
    print(f"{tramo}: flujo_sem={g['f'].mean():.0f}  trx_sem={g['t'].mean():.0f}"
          f"  conv={g['t'].sum()/g['f'].sum()*100:.2f}%")
print("Esperado: flujo 555→938, conv 10.79%→4.71%")

# Impacto de excluir rupturas
for excl, label in [(False,'Todas las tiendas'),(True,'Sin San Isidro ni Chimbote')]:
    d = dep[dep['semana'].between(2,34) & dep['nombre_tienda'].isin(comunes)]
    if excl:
        d = d[~d['nombre_tienda'].str.contains('SAN ISIDRO|CHIMBOTE')]
    r = d.groupby('anio').apply(lambda g: g['transacciones'].sum()/g['flujo'].sum())
    delta = (r[2026] - r[2025]) * 100
    print(f"{label}: 2025={r[2025]*100:.4f}%  2026={r[2026]*100:.4f}%  delta={delta:+.2f} pp")
print("Esperado: todas +0.96 pp  |  sin rupturas +1.13 pp")


## Bloque 9 · Metas comerciales

**Pregunta:** ¿Cuán exigentes son las metas 2026 y existe asociación entre cumplimiento de metas y conversión?

**Muestra:** Hoja `BD Cumplimiento de Meta de Vta`, filtrada a THE BOX con meta > 0.


In [ ]:
meta_raw = pd.read_excel(DATOS, sheet_name='BD Cumplimiento de Meta de Vta',
                        usecols=list(range(1, 11)))
meta_raw.columns = ['anio','mes','semana','categoria','nro_tienda','tienda',
                    'nombre_tienda','meta','venta_meta','cumplimiento']
for c in ['anio','semana','meta','venta_meta']:
    meta_raw[c] = pd.to_numeric(meta_raw[c], errors='coerce')

meta = meta_raw[(meta_raw['categoria']=='THE BOX') &
                meta_raw['meta'].notna() & (meta_raw['meta'] > 0)].copy()
print(f"Filas metas THE BOX: {len(meta):,}")

res_meta = meta.groupby('anio').apply(lambda g: pd.Series({
    'n':             len(g),
    'cumpl_pond':    g['venta_meta'].sum()/g['meta'].sum(),
    'pct_sobre_80':  (g['venta_meta']/g['meta'] >= 0.80).mean(),
    'pct_sobre_100': (g['venta_meta']/g['meta'] >= 1.00).mean(),
})).reset_index()
print(res_meta.to_string(index=False))
meta.to_csv(TAB / 'metas_the_box.csv', index=False)


In [ ]:
v25m = meta[(meta['anio']==2025) & meta['semana'].between(2,34)]\
    .groupby('nombre_tienda')['venta_meta'].sum().reset_index(name='v25')
v26m = meta[(meta['anio']==2026) & meta['semana'].between(2,34)]\
    .groupby('nombre_tienda').agg(meta26=('meta','sum'), v26=('venta_meta','sum')).reset_index()
exig = v25m.merge(v26m, on='nombre_tienda')

exig_pct  = exig['meta26'].sum()/exig['v25'].sum() - 1
logro_pct = exig['v26'].sum()/exig['v25'].sum() - 1
cumpl_pct = exig['v26'].sum()/exig['meta26'].sum()
print(f"Meta 2026 exige {exig_pct:+.1%} sobre venta 2025 (sem 2-34)")
print(f"Logro real 2026: {logro_pct:+.1%} sobre venta 2025")
print(f"Cumplimiento global de meta: {cumpl_pct:.1%}")

cm = meta[(meta['anio']==2026) & meta['semana'].between(2,34)]\
    .groupby('nombre_tienda').apply(
        lambda g: g['venta_meta'].sum()/g['meta'].sum()).reset_index(name='cumpl')
jj = ct[['nombre_tienda','conv']].merge(cm, on='nombre_tienda')
print(f"\nTiendas cruzadas: {len(jj)}")
r_jj, p_jj = stats.pearsonr(jj['conv'], jj['cumpl'])
print(f"Pearson conv vs cumplimiento: r={r_jj:.3f} p={p_jj:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(jj['conv']*100, jj['cumpl']*100, s=60)
for _, row in jj.iterrows():
    ax.annotate(row['nombre_tienda'].replace('THE BOX ',''),
                (row['conv']*100, row['cumpl']*100), fontsize=6, ha='left')
ax.set_xlabel('Conversión 2026 (%)'); ax.set_ylabel('Cumplimiento de meta 2026 (%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_title('Conversión vs Cumplimiento de meta por tienda (2026, sem 2-34)')
plt.tight_layout()
savefig('metas_vs_conversion.png')


## Bloque 10 · Impacto económico y sensibilidad

**Pregunta:** ¿Cuánta venta adicional representaría cerrar la brecha de conversión respecto a la frontera interna?

**Método:** Factor estacional = flujo anual 2025 / flujo semanas 2–34 de 2025. Se aplica al flujo 2026 sem 2–34 para estimar el flujo anual equivalente. Escenarios p75, p90 y máximo.

> ⚠️ **Supuestos:** factor estacional 2025 ≈ 2026; ticket promedio constante; frontera es desempeño propio observado. Las ventas potenciales **no son utilidad ni impacto garantizado**.


In [ ]:
w25 = tb[tb['anio']==2025].groupby('semana')['flujo'].sum().reset_index(name='f')
factor_est  = w25['f'].sum() / w25[w25['semana'].between(2,34)]['f'].sum()
flujo_26    = dep[(dep['anio']==2026) & dep['semana'].between(2,34)]['flujo'].sum()
flujo_anual = flujo_26 * factor_est
ticket_p    = sw26['v'].sum() / sw26['t'].sum()

print(f"Factor estacional: {factor_est:.4f}   esperado: 1.9009")
print(f"Flujo anual equiv: {flujo_anual:,.0f}   esperado: 1,697,953")
print(f"Ticket promedio:    S/{ticket_p:.2f}      esperado: S/354.90")

esc = []
for p, label in [(0.75,'p75'),(0.90,'p90'),(1.00,'máximo')]:
    v = frontera(p)
    brecha = v - conv_real
    trx_extra = flujo_anual * brecha
    venta_mm  = trx_extra * ticket_p / 1e6
    esc.append({'Escenario':label,'Frontera_%':round(v*100,4),
                'Brecha_pp':round(brecha*100,4),'Trx_extra':round(trx_extra),
                'Venta_MM_soles':round(venta_mm,2)})
    print(f"{label}: brecha={brecha*100:.4f}pp  trx_extra={trx_extra:,.0f}  S/{venta_mm:.2f}M")
print("\nEsperado p90: 1.3025pp, 22 116 trx, S/7.85M")
esc_df = pd.DataFrame(esc)
esc_df.to_csv(TAB / 'impacto_economico_escenarios.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(esc_df['Escenario'], esc_df['Venta_MM_soles'],
              color=['#55A868','#4C72B0','#DD8452'], width=0.5)
for bar, row in zip(bars, esc_df.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f"S/{row.Venta_MM_soles:.2f}M\n({row.Brecha_pp:.2f}pp)",
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Venta adicional estimada (S/ millones)')
ax.set_title('Escenarios de impacto económico por cierre de brecha de conversión\n(anualizado con factor estacional 2025)')
ax.set_ylim(0, esc_df['Venta_MM_soles'].max()*1.3)
plt.tight_layout()
savefig('impacto_escenarios.png')


## Bloque 11 · Síntesis del diagnóstico y próximos pasos

### Hallazgos por categoría epistémica

#### ✅ Verificados (calculados desde el Excel)
- Conversión real 2026 sem 2–34: ~6.5% (depurada)
- Crecimiento de venta +94%: ticket explica ~48%, flujo ~36%, conversión ~16%
- Frontera p90 ponderada por flujo: 7.78%, brecha de 1.30 pp
- Variabilidad semanal *dentro* de tienda > *entre* tiendas (SD dentro > SD entre)
- Hora 10 tiene conversión ~4.2% vs ~6.5% en el resto del día
- Domingo: mayor flujo (~22%) y conversión más baja (~5.9%)
- San Isidro: flujo aumenta ~70% entre sem 19–22 mientras trx cae, conv cae de ~11% a ~5%

#### 📊 Asociaciones estadísticas (observacionales)
- Relación negativa flujo–conversión robusta en panel hora (p<0.001) y panel semana (p≈0.06)
- Correlación negativa Spearman entre tráfico y conversión entre tiendas (r=-0.62, p=0.011)

#### 💡 Hipótesis plausibles (sin datos que las confirmen)
- Capacidad de atención como mediador de la relación tráfico–conversión
- Composición distinta de visitantes (casual vs intención de compra) según hora/día
- Incentivos de venta como driver de la conversión en semanas de alto desempeño

#### ❓ Pendiente de validación
- Dotación por hora y tienda
- Tipo de visitante (encuesta de intención)
- Auditoría de contadores de San Isidro y Chimbote
- Datos de 2026 completos para recalibrar factor estacional

---

### Matriz de hallazgos para el TFIE

| # | Hallazgo | Evidencia | Implicación TFIE | Validación pendiente |
|---|---|---|---|---|
| 1 | Conv. real 2026: ~6.5% (depurada) | Bloque 4, sw26 | Línea base del diagnóstico | Verificar cobertura de contadores |
| 2 | Crecimiento +94%: ticket lidera | Bloque 3, descomp. log. | Conversión mejoró pero no es el motor principal | Verificar composición de tiendas |
| 3 | Variabilidad dentro > entre tiendas | Bloque 6, SD y R² | Oportunidad en consistencia interna de cada local | Auditar semanas de alta vs baja conv. |
| 4 | Relación neg. flujo–conv (β≈−0.016) | Bloque 5, M1 panel | Asociación robusta, NO causalidad demostrada | Datos de dotación por hora |
| 5 | Flujo San Isidro sube 70% sem 19→22 | Bloque 8, series | Indicio de cambio de medición | Auditoría con proveedor del sensor |
| 6 | Frontera p90: 7.78%, brecha 1.30pp | Bloque 4 | Referencia interna, no meta garantizada | Verificar condiciones en sem. de alto desemp. |
| 7 | Impacto p90 anualizado: S/7.85M | Bloque 10 | Magnitud justifica inversión en diagnóstico | Factor estacional con datos 2026 completos |
| 8 | Hora 10: conv. 4.21% vs 6.54% resto | Bloque 7 | Patrón descriptivo: visitantes o dotación distintos | Registro de dotación y tipo de visitante |

---

### Próximos pasos de validación
1. **Auditar contadores** de San Isidro y Chimbote con el proveedor del sensor.
2. **Caracterizar las semanas de alta conversión** en cada tienda (condiciones operativas distintas).
3. **Recopilar datos de dotación** (staff por hora y tienda) para probar la hipótesis de capacidad.
4. **Encuesta de intención** de visita para distinguir visitantes de compra vs exploración.
5. **Completar el año 2026** para recalcular el factor estacional con datos propios.
